# Lunar Lander (continuous)

Runs 1M steps with shaped rewards:
- per-step penalty for horizontal distance from x = 0 (dist_coef = 1.0)
- built-in fuel cost reduced to 20% (keep_fuel = 0.2)
- touchdown bonus that is largest at x = 0 (centre_bonus = 150)

Up to 1M extra steps, checked every 50k steps on the plain environment. Training stops early if the score stops improving. The best checkpoint is saved separately.

In [1]:
!apt-get -qq install -y swig
!pip install -q 'gymnasium[box2d]' stable-baselines3 imageio imageio-ffmpeg

Selecting previously unselected package swig.
(Reading database ... 129802 files and directories currently installed.)
Preparing to unpack .../swig_4.2.0-2ubuntu1_amd64.deb ...
Unpacking swig (4.2.0-2ubuntu1) ...
Setting up swig (4.2.0-2ubuntu1) ...
Processing triggers for man-db (2.12.0-4build2) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 7.2 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 48.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 6.9 MB/s eta 0:00:00


In [2]:
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import imageio
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Shaped reward wrapper

The environment itself is unchanged. Only the reward used for training is shaped.

In [3]:
class CentredLander(gym.Wrapper):
    def __init__(self, env, dist_coef=1.0, keep_fuel=0.2, centre_bonus=150.0):
        super().__init__(env)
        self.dist_coef = dist_coef
        self.keep_fuel = keep_fuel
        self.centre_bonus = centre_bonus

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        x = float(obs[0])
        shaped = reward

        # Refund part of the built-in fuel cost (main 0.3, side 0.03 per frame)
        a0, a1 = float(action[0]), float(action[1])
        m_power = (np.clip(a0, 0.0, 1.0) + 1.0) * 0.5 if a0 > 0.0 else 0.0
        s_power = np.clip(abs(a1), 0.5, 1.0) if abs(a1) > 0.5 else 0.0
        builtin_fuel = 0.3 * m_power + 0.03 * s_power
        shaped += (1.0 - self.keep_fuel) * builtin_fuel

        # Distance from x = 0, every step
        shaped -= self.dist_coef * abs(x)

        # Touchdown bonus, largest at x = 0, zero beyond |x| = 0.1
        if terminated and reward > 0:
            closeness = max(0.0, 1.0 - abs(x) / 0.1)
            shaped += self.centre_bonus * closeness

        return obs, shaped, terminated, truncated, info

## Train up to 1M steps with early stopping

In [4]:
train_env = CentredLander(gym.make('LunarLander-v3', continuous=True))
eval_env = Monitor(gym.make('LunarLander-v3', continuous=True))

stop_cb = StopTrainingOnNoModelImprovement(max_no_improvement_evals=5, min_evals=4, verbose=1)
eval_cb = EvalCallback(
    eval_env,
    eval_freq=50_000,
    n_eval_episodes=20,
    best_model_save_path='best_final',
    callback_after_eval=stop_cb,
    deterministic=True,
    verbose=1,
)

model = PPO('MlpPolicy', train_env, verbose=1, seed=0)
model.learn(total_timesteps=1_000_000, reset_num_timesteps=False, callback=eval_cb)
model.save('ppo_lunar_final')
train_env.close()
eval_env.close()
print('saved ppo_lunar_final and best_final/best_model.zip')

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 109      |
|    ep_rew_mean     | -325     |
| time/              |          |
|    fps             | 571      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 122          |
|    ep_rew_mean          | -299         |
| time/                   |              |
|    fps                  | 458          |
|    iterations           | 2            |
|    time_elapsed         | 8            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0024861805 |
|    clip_fraction        | 0.014        |
|    clip_range           | 0.2          |
|    e

## Evaluate on the plain environment

Same metrics as before, on 200 seeds, for both the final model and the best checkpoint.

In [5]:
def evaluate(path, n_seeds=200):
    model = PPO.load(path)
    env = gym.make('LunarLander-v3', continuous=True)
    rewards, fuels, offsets = [], [], []
    on_pad, on_centre = 0, 0
    for seed in range(n_seeds):
        obs, _ = env.reset(seed=seed)
        done, total, fuel, r = False, 0.0, 0.0, 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            fuel += abs(float(action[0])) + abs(float(action[1]))
            obs, r, terminated, truncated, _ = env.step(action)
            total += r
            done = terminated or truncated
        rewards.append(total)
        fuels.append(fuel)
        x = abs(float(obs[0]))
        offsets.append(x)
        landed = obs[6] and obs[7] and r > 0
        if landed and x < 0.2:
            on_pad += 1
        if landed and x < 0.05:
            on_centre += 1
    env.close()
    print(f'--- {path} ({n_seeds} seeds) ---')
    print(f'mean reward {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}')
    print(f'solved (>=200): {np.mean(np.array(rewards) >= 200):.0%}')
    print(f'landed on pad (|x|<0.2): {on_pad}/{n_seeds}')
    print(f'landed on centre (|x|<0.05): {on_centre}/{n_seeds}')
    print(f'median |x| at landing: {np.median(offsets):.3f}')
    print(f'mean fuel per episode: {np.mean(fuels):.1f}\n')
    return model

evaluate('best_final/best_model')
evaluate('ppo_lunar_final')

--- best_final/best_model (200 seeds) ---
mean reward 217.5 +/- 75.7
solved (>=200): 84%
landed on pad (|x|<0.2): 99/200
landed on centre (|x|<0.05): 18/200
median |x| at landing: 0.179
mean fuel per episode: 311.6

--- ppo_lunar_final (200 seeds) ---
mean reward 222.0 +/- 69.5
solved (>=200): 85%
landed on pad (|x|<0.2): 88/200
landed on centre (|x|<0.05): 20/200
median |x| at landing: 0.205
mean fuel per episode: 295.7



## Record one landing from the best checkpoint

In [6]:
model = PPO.load('best_final/best_model')
env = gym.make('LunarLander-v3', continuous=True, render_mode='rgb_array')
frames = []
obs, _ = env.reset(seed=0)
done = False
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, r, terminated, truncated, _ = env.step(action)
    frames.append(env.render())
    done = terminated or truncated
env.close()

os.makedirs('videos', exist_ok=True)
imageio.mimsave('videos/final_landing.mp4', frames, fps=50, macro_block_size=1)
print('saved videos/final_landing.mp4')

saved videos/final_landing.mp4


## Test models under different conditions for robustness

In [9]:
models = {
    "best_final (scratch, best ckpt)": "best_final/best_model",
    "ppo_lunar_final (scratch, last)": "ppo_lunar_final",
}
conditions = [
    ("calm", dict(enable_wind=False)),
    ("wind 5", dict(enable_wind=True, wind_power=5.0)),
    ("wind 10", dict(enable_wind=True, wind_power=10.0)),
    ("wind 15", dict(enable_wind=True, wind_power=15.0)),
]

for name, path in models.items():
    print(f"=== {name} ===")
    for cond_name, kwargs in conditions:
        r = evaluate(path, **kwargs)
        print(f"{cond_name:8s} | mean {r['mean']:6.1f} +/- {r['std']:5.1f} | "
              f"solved {r['solved']:.0%} | pad {r['pad']:3d}/200 | "
              f"centre {r['centre']:3d}/200 | median|x| {r['median_x']:.3f} | "
              f"fuel {r['fuel']:.1f}")
    print()

=== best_final (scratch, best ckpt) ===
calm     | mean  217.5 +/-  75.7 | solved 84% | pad  99/200 | centre  18/200 | median|x| 0.179 | fuel 311.6
wind 5   | mean  216.8 +/-  76.1 | solved 84% | pad  95/200 | centre  24/200 | median|x| 0.191 | fuel 340.1
wind 10  | mean  197.0 +/-  99.5 | solved 72% | pad  90/200 | centre  16/200 | median|x| 0.188 | fuel 328.8
wind 15  | mean  185.7 +/- 108.9 | solved 65% | pad  82/200 | centre  20/200 | median|x| 0.190 | fuel 365.1

=== ppo_lunar_final (scratch, last) ===
calm     | mean  222.0 +/-  69.5 | solved 85% | pad  88/200 | centre  20/200 | median|x| 0.205 | fuel 295.7
wind 5   | mean  208.2 +/-  87.4 | solved 79% | pad  77/200 | centre  18/200 | median|x| 0.226 | fuel 314.2
wind 10  | mean  205.0 +/-  94.3 | solved 78% | pad  74/200 | centre  17/200 | median|x| 0.244 | fuel 304.0
wind 15  | mean  182.6 +/- 109.8 | solved 61% | pad  62/200 | centre  11/200 | median|x| 0.299 | fuel 300.3

